## Session 3 — Embeddings + RAG Part 1

**XRO Tech · AI + Google Cloud Architecture Program · Batch 2**

By the end of this session you'll have a searchable vector index stored in your GCS folder — the foundation that makes RAG possible.

Run every cell top to bottom. Read the markdown above each cell before running it.

**What you'll do:**

| Cell | Task |
|---|---|
| 1 | Check your environment |
| 2 | Connect to the embedding model |
| 3 | Convert text into a vector |
| 4 | Measure similarity between sentences |
| 5 | Split a document into overlapping chunks |
| 6 | Build a full vector index |
| 7 | Save the index to GCS for Session 4

**Cell 1 — Environment check.**

In [1]:
import os

STUDENT_ID = os.environ.get("STUDENT_ID", "NOT_SET")
BATCH_ID = os.environ.get("BATCH_ID", "NOT_SET")
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "NOT_SET")
REGION = "asia-south1"
GCS_BUCKET = "xro-lab-data"
MY_FOLDER = f"{BATCH_ID}/{STUDENT_ID}/"

print(f"Student ID  : {STUDENT_ID}")
print(f"Batch ID    : {BATCH_ID}")
print(f"GCP Project : {PROJECT_ID}")
print(f"Region      : {REGION} (Mumbai)")
print(f"My GCS path : gs://{GCS_BUCKET}/{MY_FOLDER}")

Student ID  : s02
Batch ID    : batch2
GCP Project : xro-lab
Region      : asia-south1 (Mumbai)
My GCS path : gs://xro-lab-data/batch2/s02/


**Cell 2 — Connect to the embedding model.** `text-embedding-004` converts text into a 768-number vector that captures its meaning, not just its words.

In [2]:
import vertexai
from vertexai.language_models import TextEmbeddingModel
from google.cloud import storage
import numpy as np
import json
import time

vertexai.init(project=PROJECT_ID, location=REGION)
embedding_model = TextEmbeddingModel.from_pretrained("text-embedding-004")

print("Connected to text-embedding-004")
print("Vector dimension: 768 floats per text input")

Connected to text-embedding-004
Vector dimension: 768 floats per text input


/usr/local/lib/python3.11/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


**Cell 3 — Your first embedding.** Similar texts produce similar vectors. This is how the model compares meaning, not just matching words.

The embedding API has a rate limit shared across the whole batch. `embed_text()` below retries automatically with a short wait if it hits a temporary quota error -- this is expected occasionally when many students call the API at the same time, and is not a sign anything is broken.

In [3]:
from google.api_core.exceptions import ResourceExhausted


def embed_text(text: str, max_retries: int = 5) -> list[float]:
    """Convert text to a 768-dimension vector using Vertex AI, retrying on rate limits."""
    for attempt in range(max_retries):
        try:
            embeddings = embedding_model.get_embeddings([text])
            return embeddings[0].values
        except ResourceExhausted:
            wait_time = 2 ** attempt
            print(f"  Rate limit hit, waiting {wait_time}s before retry {attempt + 1}/{max_retries}...")
            time.sleep(wait_time)
    raise RuntimeError("Embedding failed after multiple retries. Wait a minute and try again.")


t0 = time.time()
test_sentence = "Artificial intelligence is transforming industries in Assam and Northeast India."
vector = embed_text(test_sentence)
elapsed = round(time.time() - t0, 2)

print(f"Input  : {test_sentence}")
print(f"Output : [{vector[0]:.4f}, {vector[1]:.4f}, {vector[2]:.4f}, ... {vector[-1]:.4f}]")
print(f"Length : {len(vector)} dimensions")
print(f"Time   : {elapsed}s")

Input  : Artificial intelligence is transforming industries in Assam and Northeast India.
Output : [0.0169, -0.0349, 0.0058, ... -0.0439]
Length : 768 dimensions
Time   : 0.43s


**Cell 4 — Cosine similarity.** This measures how close two vectors are in meaning. A score near 1.0 means very similar, near 0.0 means unrelated.

In [4]:
def cosine_similarity(v1: list[float], v2: list[float]) -> float:
    """Compare two vectors. Returns 1.0 if identical, 0.0 if unrelated."""
    a, b = np.array(v1), np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


pairs = [
    ("Artificial intelligence is transforming industries.",
     "Machine learning is changing how businesses operate.",
     "SIMILAR -- both about AI/ML"),
    ("Artificial intelligence is transforming industries.",
     "The monsoon season in Assam brings heavy rainfall.",
     "DIFFERENT -- unrelated topics"),
    ("Cloud computing allows remote data storage.",
     "GCP provides infrastructure for storing files online.",
     "SIMILAR -- both about cloud storage"),
    ("Python is a programming language.",
     "I enjoy eating mangoes in summer.",
     "DIFFERENT -- code vs food"),
]

for text_a, text_b, label in pairs:
    vec_a = embed_text(text_a)
    time.sleep(0.5)
    vec_b = embed_text(text_b)
    time.sleep(0.5)
    score = cosine_similarity(vec_a, vec_b)
    print(f"{label}")
    print(f"  A: {text_a}")
    print(f"  B: {text_b}")
    print(f"  Score: {score:.4f}")
    print()

SIMILAR -- both about AI/ML
  A: Artificial intelligence is transforming industries.
  B: Machine learning is changing how businesses operate.
  Score: 0.7181

DIFFERENT -- unrelated topics
  A: Artificial intelligence is transforming industries.
  B: The monsoon season in Assam brings heavy rainfall.
  Score: 0.2158

SIMILAR -- both about cloud storage
  A: Cloud computing allows remote data storage.
  B: GCP provides infrastructure for storing files online.
  Score: 0.6600

  Rate limit hit, waiting 1s before retry 1/5...
  Rate limit hit, waiting 2s before retry 2/5...
  Rate limit hit, waiting 4s before retry 3/5...
  Rate limit hit, waiting 8s before retry 4/5...
  Rate limit hit, waiting 16s before retry 5/5...


RuntimeError: Embedding failed after multiple retries. Wait a minute and try again.

**Cell 5 — Document chunking.** RAG needs to split documents into overlapping pieces. Too large and the model gets irrelevant context; too small and chunks lose their meaning. Overlap keeps context from breaking at chunk boundaries.

The sample document below stands in for something like a PDF or report you'd index for real.

In [5]:
def chunk_document(text: str, chunk_size: int = 60, overlap: int = 8) -> list[str]:
    """Split text into overlapping chunks of roughly chunk_size words."""
    words = text.split()
    chunks = []
    start_idx = 0

    while start_idx < len(words):
        end_idx = start_idx + chunk_size
        chunk = " ".join(words[start_idx:end_idx])
        chunks.append(chunk)
        start_idx = end_idx - overlap

    return [c for c in chunks if len(c.strip()) > 20]


SAMPLE_DOCUMENT = """
Artificial intelligence is transforming industries across Northeast India. In Assam,
several startups are using machine learning models to improve agricultural yield
prediction. By analysing satellite imagery and weather patterns, these systems can
forecast crop health with 85% accuracy. Farmers in Bodoland are among the early
adopters of these AI tools, accessing them via mobile applications.

Google Cloud Platform provides the infrastructure for these AI systems. Vertex AI
allows engineers to build, train and deploy machine learning models in a managed
environment. Cloud Storage provides durable object storage for training datasets
and model artifacts. Cloud Run deploys containerised applications that scale
automatically to demand.

Retrieval Augmented Generation, or RAG, is a technique that combines the power of
large language models with the ability to search private document collections.
Instead of fine-tuning a model on your data, RAG embeds your documents as vectors,
stores them in a vector index, and retrieves the most relevant chunks at query time.
This allows the language model to answer questions using your private documents
without hallucinating information.

Text embeddings are the foundation of RAG. The text-embedding-004 model from
Google converts any text into a 768-dimensional vector that captures its semantic
meaning. Similar texts produce vectors that are close together in this
768-dimensional space. Distance between vectors, measured using cosine similarity,
determines how semantically related two pieces of text are.
"""

chunks = chunk_document(SAMPLE_DOCUMENT)

print(f"Document: {len(SAMPLE_DOCUMENT.split())} words")
print(f"Chunks  : {len(chunks)} (size=60 words, overlap=8 words)")
print()
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1} ({len(chunk.split())} words):")
    print(f"  {chunk[:120]}...")
    print()

Document: 223 words
Chunks  : 5 (size=60 words, overlap=8 words)

Chunk 1 (60 words):
  Artificial intelligence is transforming industries across Northeast India. In Assam, several startups are using machine ...

Chunk 2 (60 words):
  via mobile applications. Google Cloud Platform provides the infrastructure for these AI systems. Vertex AI allows engine...

Chunk 3 (60 words):
  Augmented Generation, or RAG, is a technique that combines the power of large language models with the ability to search...

Chunk 4 (60 words):
  This allows the language model to answer questions using your private documents without hallucinating information. Text ...

Chunk 5 (15 words):
  between vectors, measured using cosine similarity, determines how semantically related two pieces of text are....



**Cell 6 — Build the full vector index.** This is the core operation behind RAG: convert every chunk into a vector and keep the chunk and its vector together.

In [6]:
print("Building vector index...")
print()

vector_index = []
for i, chunk in enumerate(chunks):
    vector = embed_text(chunk)
    vector_index.append({
        "chunk_id": i,
        "text": chunk,
        "vector": vector,
        "char_count": len(chunk),
        "word_count": len(chunk.split()),
    })
    print(f"  [{i + 1}/{len(chunks)}] Embedded chunk {i} -- {len(chunk.split())} words")
    time.sleep(0.5)

print()
print(f"Vector index complete: {len(vector_index)} chunks, {len(vector_index[0]['vector'])} dims each")


def search_index(query: str, top_k: int = 3) -> list[dict]:
    """Find the most relevant chunks for a query."""
    query_vector = embed_text(query)
    results = []
    for item in vector_index:
        score = cosine_similarity(query_vector, item["vector"])
        results.append({**item, "score": score})
    return sorted(results, key=lambda x: x["score"], reverse=True)[:top_k]


print()
print("Quick search test: 'What is RAG?'")
results = search_index("What is RAG and how does it work?")
for r in results:
    print(f"Score {r['score']:.4f}: {r['text'][:100]}...")

Building vector index...

  [1/5] Embedded chunk 0 -- 60 words
  [2/5] Embedded chunk 1 -- 60 words
  [3/5] Embedded chunk 2 -- 60 words
  [4/5] Embedded chunk 3 -- 60 words
  [5/5] Embedded chunk 4 -- 15 words

Vector index complete: 5 chunks, 768 dims each

Quick search test: 'What is RAG?'
Score 0.6498: Augmented Generation, or RAG, is a technique that combines the power of large language models with t...
Score 0.5926: This allows the language model to answer questions using your private documents without hallucinatin...
Score 0.5785: via mobile applications. Google Cloud Platform provides the infrastructure for these AI systems. Ver...


**Cell 7 — Save your vector index to GCS.** The index lives in your personal GCS folder. Session 4 loads this exact file to build the full RAG query pipeline.

In [7]:
gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(GCS_BUCKET)
index_path = f"{MY_FOLDER}session3_vector_index.json"

index_data = {
    "metadata": {
        "student_id": STUDENT_ID,
        "batch_id": BATCH_ID,
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "model": "text-embedding-004",
        "dimensions": 768,
        "chunk_count": len(vector_index),
        "chunk_size": 60,
        "overlap": 8,
    },
    "chunks": vector_index,
}

blob = bucket.blob(index_path)
blob.upload_from_string(
    json.dumps(index_data, indent=2),
    content_type="application/json",
)

print("Vector index saved")
print(f"  Path   : gs://{GCS_BUCKET}/{index_path}")
print(f"  Chunks : {len(vector_index)}")
print(f"  Size   : ~{blob.size // 1024} KB")

Vector index saved
  Path   : gs://xro-lab-data/batch2/s02/session3_vector_index.json
  Chunks : 5
  Size   : ~115 KB


---

### Session 3 complete

You built the search engine at the heart of every RAG system: embeddings, similarity scoring, document chunking, and a saved vector index.

**Before you close:** save the notebook, and commit to GitHub if you're tracking your work there.

**Next session:** Session 4 loads this vector index, builds the full RAG query pipeline, and deploys it to Cloud Run with a permanent URL.